<a href="https://colab.research.google.com/github/Leanhchudang2511/baitaptrituenhantao/blob/main/GPA_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎓 GPA Prediction App — NCKH UEH
Chạy từng cell theo thứ tự để khởi động app Streamlit ngay trong Colab.

In [1]:
!pip install streamlit pyngrok scikit-learn openpyxl seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 60.4 MB/s eta 0:00:00
✅ Cài xong!


In [2]:
from google.colab import files
uploaded = files.upload()

📂 Chọn file gpaueh.xlsx để upload...


Saving gpaueh.xlsx to gpaueh.xlsx
✅ Upload xong!


In [3]:
app_code = r"""
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

st.set_page_config(page_title="Du Doan GPA", page_icon="🎓", layout="wide")
st.markdown('''
<style>
.main-header{background:linear-gradient(135deg,#1a237e,#3949ab);padding:2rem;border-radius:16px;text-align:center;margin-bottom:2rem}
.main-header h1{color:white;font-size:2rem;margin:0}
.main-header p{color:#c5cae9;margin-top:.5rem}
.pred-box{background:linear-gradient(135deg,#e8eaf6,#c5cae9);border-radius:16px;padding:2rem;text-align:center;border:2px solid #3949ab}
.stButton>button{background:linear-gradient(135deg,#1a237e,#3949ab);color:white;border:none;border-radius:10px;padding:.7rem 2rem;font-size:1.05rem;width:100%}
</style>
''', unsafe_allow_html=True)

FEATURES = ["gio_hoc","so_mon","lam_them","gio_ngu","clb","chuyen_can","hoc_nhom","mxh","nam_hoc"]
FEAT_LABELS = ["Gio hoc/tuan","So mon","Lam them","Gio ngu","CLB","Chuyen can(%)","Hoc nhom","MXH(h/ngay)","Nam hoc"]

@st.cache_data
def load_data(uploaded_path=None):
    paths = [uploaded_path, "gpaueh.xlsx", "/content/gpaueh.xlsx"]
    for p in paths:
        if p and os.path.exists(p):
            try:
                raw = pd.read_excel(p)
                c = raw.columns.tolist()
                df = pd.DataFrame()
                df["GPA"]        = pd.to_numeric(raw[c[1]], errors="coerce")
                df["nam_hoc"]    = raw[c[2]].astype(str).str.extract(r"(\d+)").astype(float).fillna(1)
                df["gio_hoc"]    = pd.to_numeric(raw[c[5]], errors="coerce")
                df["so_mon"]     = pd.to_numeric(raw[c[6]], errors="coerce")
                df["lam_them"]   = ((raw[c[7]].astype(str).str.strip()=="Co")|(raw[c[7]].astype(str).str.strip()=="Có")).astype(int)
                df["gio_ngu"]    = pd.to_numeric(raw[c[8]], errors="coerce")
                df["clb"]        = ((raw[c[9]].astype(str).str.strip()=="Co")|(raw[c[9]].astype(str).str.strip()=="Có")).astype(int)
                df["chuyen_can"] = pd.to_numeric(raw[c[10]], errors="coerce")
                df["hoc_nhom"]   = (raw[c[11]].astype(str).str.strip()=="Hoc nhom").astype(int)
                df["mxh"]        = pd.to_numeric(raw[c[12]], errors="coerce")
                df = df.dropna()
                if len(df) > 0:
                    return df, True
            except Exception:
                pass
    np.random.seed(42); n=291
    gh=np.random.randint(5,50,n); sm=np.random.randint(3,8,n)
    lt=np.random.choice([0,1],n,p=[0.55,0.45]); gn=np.random.uniform(4,10,n)
    cl=np.random.choice([0,1],n,p=[0.4,0.6]); cc=np.random.uniform(50,100,n)
    hn=np.random.choice([0,1],n); mx=np.random.uniform(0,10,n); nh=np.random.choice([1,2,3,4],n)
    gpa=np.clip(0.03*gh+0.05*cc/10-0.15*lt+0.08*gn+0.05*cl+0.04*hn-0.05*mx+0.02*nh+np.random.normal(0,0.2,n)+1.5,2.0,4.0).round(2)
    return pd.DataFrame({"GPA":gpa,"gio_hoc":gh,"so_mon":sm,"lam_them":lt,"gio_ngu":gn.round(1),
                          "clb":cl,"chuyen_can":cc.round(1),"hoc_nhom":hn,"mxh":mx.round(1),"nam_hoc":nh}), False

@st.cache_resource
def train_models(n_rows):
    df, _ = load_data()
    X = df[FEATURES].values; y = df["GPA"].values
    scaler = StandardScaler(); X_sc = scaler.fit_transform(X)
    X_tr,X_te,y_tr,y_te = train_test_split(X_sc,y,test_size=0.2,random_state=42)
    kf = KFold(n_splits=5,shuffle=True,random_state=42)
    models_def = [
        ("Linear Regression", LinearRegression()),
        ("Ridge",             Ridge(alpha=1.0)),
        ("Lasso",             Lasso(alpha=0.01)),
        ("ElasticNet",        ElasticNet(alpha=0.01,l1_ratio=0.5)),
        ("Random Forest",     RandomForestRegressor(n_estimators=200,random_state=42)),
        ("Gradient Boosting", GradientBoostingRegressor(n_estimators=200,learning_rate=0.05,random_state=42)),
        ("SVR (rbf)",         SVR(kernel="rbf",C=1.0,epsilon=0.1)),
    ]
    results = {}
    for name, m in models_def:
        m.fit(X_tr,y_tr); pred=np.clip(m.predict(X_te),2.0,4.0)
        cv=cross_val_score(m,X_sc,y,cv=kf,scoring="r2").mean()
        results[name]={"model":m,"r2":r2_score(y_te,pred),"mae":mean_absolute_error(y_te,pred),
                       "rmse":np.sqrt(mean_squared_error(y_te,pred)),"cv_r2":cv,"pred":pred,"y_te":y_te}
    return results, scaler

with st.sidebar:
    st.markdown("### ⚙️ Cai dat")
    uploaded = st.file_uploader("📂 Upload gpaueh.xlsx", type=["xlsx","xls"])
    up_path = None
    if uploaded:
        up_path = "/tmp/gpa_upload.xlsx"
        with open(up_path,"wb") as f:
            f.write(uploaded.read())
        st.cache_data.clear()
        st.cache_resource.clear()

df, real_data = load_data(up_path)

with st.spinner("🔄 Huan luyen mo hinh..."):
    results, scaler = train_models(len(df))

best_name = max(results, key=lambda k: results[k]["r2"])

with st.sidebar:
    st.markdown("### 🤖 Chon mo hinh")
    model_choice = st.selectbox("Mo hinh:", list(results.keys()), index=list(results.keys()).index(best_name))
    ch = results[model_choice]
    st.markdown(f"**R2:** `{ch['r2']:.4f}`")
    st.markdown(f"**MAE:** `{ch['mae']:.4f}`")
    st.markdown(f"**CV R2:** `{ch['cv_r2']:.4f}`")
    st.markdown("---")
    data_label = "✅ Excel thuc" if real_data else "🧪 Du lieu mau"
    st.markdown(f"📊 **{len(df)} sinh vien** · {data_label}")

st.markdown('''
<div class="main-header">
  <h1>🎓 Du Doan GPA Sinh Vien</h1>
  <p>Nhap thoi quen hoc tap de du doan GPA ky toi · NCKH UEH 2025</p>
</div>
''', unsafe_allow_html=True)

tab1, tab2, tab3 = st.tabs(["🔮 Du Doan GPA", "📊 Phan Tich Du Lieu", "🏆 So Sanh Mo Hinh"])

with tab1:
    col1, col2 = st.columns([1.2,1], gap="large")
    with col1:
        st.markdown("#### 📝 Nhap thong tin hoc tap")
        ca, cb = st.columns(2)
        with ca:
            nam_hoc    = st.selectbox("Nam hoc", [1,2,3,4], format_func=lambda x: f"Nam {x}")
            gio_hoc    = st.slider("⏱ Gio hoc/tuan", 1, 80, 20)
            so_mon     = st.slider("📚 So mon", 1, 10, 5)
            chuyen_can = st.slider("✅ Chuyen can (%)", 50, 100, 80)
        with cb:
            gio_ngu  = st.slider("😴 Gio ngu/dem", 3.0, 12.0, 7.0, 0.5)
            mxh      = st.slider("📱 MXH (gio/ngay)", 0.0, 16.0, 3.0, 0.5)
            lam_them = st.radio("💼 Lam them?", ["Khong","Co"], horizontal=True)
            clb      = st.radio("🏃 Tham gia CLB?", ["Khong","Co"], horizontal=True)
            hoc_nhom = st.radio("👥 Hinh thuc hoc?", ["Tu hoc","Hoc nhom"], horizontal=True)
        btn = st.button("🔮 Du Doan GPA", use_container_width=True)

    with col2:
        st.markdown("#### 📈 Ket qua du doan")
        if btn:
            inp = np.array([[gio_hoc, so_mon,
                             1 if lam_them=="Co" else 0,
                             gio_ngu,
                             1 if clb=="Co" else 0,
                             chuyen_can,
                             1 if hoc_nhom=="Hoc nhom" else 0,
                             mxh, nam_hoc]])
            gpa_pred = float(np.clip(results[model_choice]["model"].predict(scaler.transform(inp)), 2.0, 4.0))
            if gpa_pred >= 3.6:   band, color, emoji = "Xuat sac", "#1b5e20", "🌟"
            elif gpa_pred >= 3.2: band, color, emoji = "Gioi",     "#2e7d32", "✨"
            elif gpa_pred >= 2.5: band, color, emoji = "Kha",      "#e65100", "👍"
            else:                 band, color, emoji = "Trung binh","#b71c1c", "📖"
            pred_html = (
                f'<div class="pred-box">'
                f'<p style="color:#555">{emoji} GPA du doan</p>'
                f'<h2 style="color:{color};font-size:4rem;margin:0">{gpa_pred:.2f}</h2>'
                f'<p style="color:{color};font-weight:bold;font-size:1.2rem">{band}</p>'
                f'<p style="color:#888;font-size:.85rem">Mo hinh: {model_choice}</p>'
                f'</div>'
            )
            st.markdown(pred_html, unsafe_allow_html=True)
            cats  = ["Gio hoc","Chuyen can","Gio ngu","MXH(dao)","So mon"]
            vals  = [gio_hoc/80, chuyen_can/100, gio_ngu/12, 1-mxh/16, so_mon/10]
            angles = np.linspace(0, 2*np.pi, 5, endpoint=False).tolist()
            vals_r  = vals + [vals[0]]
            angles_r = angles + [angles[0]]
            fig, ax = plt.subplots(figsize=(4,4), subplot_kw=dict(polar=True))
            ax.fill(angles_r, vals_r, alpha=0.25, color="#3949ab")
            ax.plot(angles_r, vals_r, "o-", lw=2, color="#3949ab")
            ax.set_thetagrids(np.degrees(angles), cats, fontsize=8)
            ax.set_ylim(0, 1)
            ax.set_title("Ho so hoc tap", fontsize=10, fontweight="bold", pad=15)
            fig.patch.set_alpha(0)
            st.pyplot(fig, use_container_width=True)
            plt.close()
            tips = []
            if gio_hoc < 15:    tips.append("📚 Tang gio tu hoc len ≥15h/tuan")
            if chuyen_can < 80: tips.append("✅ Cai thien chuyen can len >80%")
            if mxh > 4:         tips.append("📱 Giam MXH xuong <4h/ngay")
            if gio_ngu < 6:     tips.append("😴 Ngu du 6-8h/dem")
            if not tips:        tips.append("🌟 Thoi quen hoc tap cua ban rat tot!")
            st.markdown("**💡 Goi y:**")
            for t in tips:
                st.markdown(f"- {t}")
        else:
            st.info("👈 Nhap thong tin va nhan **Du Doan GPA**")

with tab2:
    st.markdown("#### 📊 Phan Tich Du Lieu")
    m1,m2,m3,m4 = st.columns(4)
    m1.metric("So sinh vien", len(df))
    m2.metric("GPA trung binh", f"{df['GPA'].mean():.2f}")
    m3.metric("GPA cao nhat",   f"{df['GPA'].max():.2f}")
    m4.metric("GPA thap nhat",  f"{df['GPA'].min():.2f}")
    c1, c2 = st.columns(2)
    with c1:
        fig, ax = plt.subplots(figsize=(6,4))
        ax.hist(df["GPA"], bins=25, color="#3949ab", alpha=0.8, edgecolor="white")
        ax.axvline(df["GPA"].mean(), color="#e53935", ls="--", lw=2, label=f"TB={df['GPA'].mean():.2f}")
        ax.set_title("Phan bo GPA", fontweight="bold")
        ax.legend(); ax.grid(axis="y", alpha=0.3)
        fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()
    with c2:
        corr = df[FEATURES+["GPA"]].corr()
        fig, ax = plt.subplots(figsize=(6,4))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", ax=ax, center=0,
                    annot_kws={"size":7},
                    xticklabels=FEAT_LABELS+["GPA"],
                    yticklabels=FEAT_LABELS+["GPA"])
        ax.set_title("Ma tran tuong quan", fontweight="bold")
        fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()
    c3, c4 = st.columns(2)
    with c3:
        fig, ax = plt.subplots(figsize=(6,4))
        ax.scatter(df["gio_hoc"], df["GPA"], alpha=0.4, color="#e76f51", s=25)
        z = np.polyfit(df["gio_hoc"], df["GPA"], 1); p = np.poly1d(z)
        xs = np.linspace(df["gio_hoc"].min(), df["gio_hoc"].max(), 100)
        ax.plot(xs, p(xs), "b-", lw=2)
        ax.set_title("Gio hoc/tuan vs GPA", fontweight="bold")
        ax.grid(alpha=0.3); fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()
    with c4:
        fi  = results["Random Forest"]["model"].feature_importances_
        idx = np.argsort(fi)
        fig, ax = plt.subplots(figsize=(6,4))
        colors_bar = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(FEAT_LABELS)))
        ax.barh([FEAT_LABELS[i] for i in idx], fi[idx],
                color=colors_bar[np.argsort(idx)], edgecolor="white", height=0.6)
        ax.set_title("Feature Importance", fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
        for i, v in enumerate(fi[idx]):
            ax.text(v+.002, i, f"{v:.3f}", va="center", fontsize=9)
        fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()

with tab3:
    st.markdown("#### 🏆 So Sanh Mo Hinh")
    rows = [{"Mo hinh":n, "R2 (test)":round(r["r2"],4), "MAE":round(r["mae"],4),
             "RMSE":round(r["rmse"],4), "CV R2":round(r["cv_r2"],4)} for n,r in results.items()]
    comp_df = pd.DataFrame(rows).sort_values("R2 (test)", ascending=False)
    def hl(s):
        best = s==s.max() if s.name in ["R2 (test)","CV R2"] else s==s.min()
        return ["background-color:#c8e6c9;font-weight:bold" if v else "" for v in best]
    st.dataframe(comp_df.style.apply(hl, subset=["R2 (test)","MAE","RMSE","CV R2"]),
                 use_container_width=True, hide_index=True)
    c1, c2 = st.columns(2)
    names = list(results.keys())
    r2s   = [results[n]["r2"] for n in names]
    with c1:
        fig, ax = plt.subplots(figsize=(6,4))
        cols2 = ["#1b5e20" if r==max(r2s) else "#4A90D9" for r in r2s]
        ax.barh(names, r2s, color=cols2, edgecolor="white", height=0.6)
        ax.set_title("R2 Score", fontweight="bold")
        ax.grid(axis="x", alpha=0.3)
        for i, v in enumerate(r2s):
            ax.text(max(v+.005, .01), i, f"{v:.4f}", va="center", fontsize=8)
        fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()
    with c2:
        br = results[best_name]
        fig, ax = plt.subplots(figsize=(6,4))
        ax.scatter(br["y_te"], br["pred"], alpha=0.6, color="#3949ab", s=40)
        mn2 = min(br["y_te"].min(), br["pred"].min())
        mx2 = max(br["y_te"].max(), br["pred"].max())
        ax.plot([mn2,mx2], [mn2,mx2], "r--", lw=2, label="Ly tuong")
        ax.set_xlabel("GPA Thuc te")
        ax.set_ylabel("GPA Du doan")
        ax.set_title(f"Actual vs Predicted ({best_name})", fontweight="bold")
        ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout(); st.pyplot(fig, use_container_width=True); plt.close()
    best_r2  = results[best_name]['r2']
    best_mae = results[best_name]['mae']
    st.info(f"💡 Mo hinh tot nhat: **{best_name}** — R2={best_r2:.4f}, MAE={best_mae:.4f}")
"""

with open('streamlit_app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

✅ Đã tạo streamlit_app.py


In [4]:

from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = '3DFvCJVYwIGdAyqwiLEDuaBONez_29crAqnArhG8sbtsvXpC3'
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

proc = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py',
     '--server.port=8501', '--server.headless=true',
     '--server.enableCORS=false', '--server.enableXsrfProtection=false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)
public_url = ngrok.connect(8501)

🎓 APP ĐANG CHẠY TẠI: NgrokTunnel: "https://bruising-slideshow-chapped.ngrok-free.dev" -> "http://localhost:8501"
(Nhấn Ctrl+C hoặc stop cell để tắt app)
